In [1]:
import csv
import json
import pandas as pd
import numpy as np
import requests

In [2]:
import csv
import json

accepted = []
rejected = []

In [3]:
def parse_row(row):

    row["age"] = int(row["age"])
    row["fees"] = int(row["fees"])
    row["previous_no_shows"] = int(row["previous_no_shows"])
    row["distance"] = float(row["distance"])

    return row

In [8]:
with open("appointments.csv", "r") as file:

    reader = csv.DictReader(file)

    for row in reader:

        try:
            clean_row = parse_row(row)
            accepted.append(clean_row)

        except (ValueError, TypeError, KeyError) as e:
            rejected.append({
                "row": row,
                "error": str(e)
            })

In [10]:
with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=4)

In [11]:
total_rows = len(accepted) + len(rejected)

print("Total rows:", total_rows)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

Total rows: 1020
Accepted: 0
Rejected: 1020


In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/appointments.csv")

print(df.shape)
df.info()

print(df.isnull().sum())
print(df.duplicated().sum())

(1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on      

In [13]:
df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

After removing duplicates: (1000, 14)


In [14]:
df["booking_channel"] = (
    df["booking_channel"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["booking_channel"] = df["booking_channel"].replace({
    "app": "App",
    "mobile app": "App",
    "phone": "Phone",
    "walkin": "Walk-In",
    "walk-in": "Walk-In"
})

In [15]:
df["appointment_date"] = pd.to_datetime(
    df["appointment_date"],
    errors="coerce",
    dayfirst=True
)

KeyError: 'appointment_date'

In [16]:
print(df.columns.tolist())

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [17]:
df.columns = df.columns.str.strip()

In [18]:
print(df.columns.tolist())

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [19]:
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce",
    dayfirst=True
)

/tmp/ipykernel_3002/1133481241.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["appt_date"] = pd.to_datetime(


In [21]:
df = df.dropna(subset=["appt_date"])

In [23]:
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce",
    dayfirst=True
)

/tmp/ipykernel_3002/1133481241.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["appt_date"] = pd.to_datetime(


In [24]:
df = df.dropna(subset=["appt_date"])

print("After removing invalid dates:", df.shape)

After removing invalid dates: (994, 14)


In [25]:
df = df.dropna(subset=["status"])

print("After removing missing statuses:", df.shape)

After removing missing statuses: (990, 14)


In [26]:
print("Final dataset shape:", df.shape)

print(df.isnull().sum())

print(df.head())

Final dataset shape: (990, 14)
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           0
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              0
dtype: int64
  appointment_id patient_id clinic_id   booked_on  appt_date   age gender  \
0          A0001       P509       C05  2025-01-24 2025-01-25  60.0      F   
1          A0002       P046       C07  2025-05-27 2025-06-06  39.0      M   
2          A0003       P009       C02  2025-03-24 2025-03-30  63.0      F   
4          A0004       P482       C01  2025-04-18 2025-04-22  52.0      F   
5          A0005       P251       C01  2024-11-23 2025-01-02  64.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In    

In [27]:
df.to_csv("cleaned_appointments.csv", index=False)

In [28]:
print(df.columns.tolist())

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/cleaned_appointments.csv")

In [30]:
invalid_ages = ((df["age"] < 1) | (df["age"] > 100)).sum()

print("Invalid ages:", invalid_ages)

df.loc[
    (df["age"] < 1) | (df["age"] > 100),
    "age"
] = np.nan

Invalid ages: 5


In [31]:
median_age = df["age"].median()

print("Median age:", median_age)

df["age"] = df["age"].fillna(median_age)

Median age: 48.0


In [32]:
median_distance = df["distance_km"].median()

print("Median distance:", median_distance)

df["distance_km"] = df["distance_km"].fillna(median_distance)

Median distance: 8.25


In [33]:
Q1 = df["distance_km"].quantile(0.25)
Q3 = df["distance_km"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

distance_capped = (
    (df["distance_km"] < lower) |
    (df["distance_km"] > upper)
).sum()

print("Distance values capped:", distance_capped)

df["distance_km"] = np.clip(
    df["distance_km"],
    lower,
    upper
)

Distance values capped: 26


In [34]:
Q1 = df["fee_inr"].quantile(0.25)
Q3 = df["fee_inr"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

fee_capped = (
    (df["fee_inr"] < lower) |
    (df["fee_inr"] > upper)
).sum()

print("Fee values capped:", fee_capped)

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower,
    upper
)

Fee values capped: 3


In [35]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round(0).astype(int)

In [36]:
print("Invalid ages:", invalid_ages)
print("Median age:", median_age)
print("Median distance:", median_distance)
print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

print("\nFinal dataset shape:", df.shape)

print("\nMissing values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

Invalid ages: 5
Median age: 48.0
Median distance: 8.25
Distance values capped: 26
Fee values capped: 3

Final dataset shape: (990, 14)

Missing values:
age            0
distance_km    0
fee_inr        0
dtype: int64


In [37]:
df.to_csv("cleaned_appointments.csv", index=False)

In [38]:
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce",
    dayfirst=True
)

df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce",
    dayfirst=True
)

/tmp/ipykernel_3002/2545466072.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["booked_on"] = pd.to_datetime(
/tmp/ipykernel_3002/2545466072.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["appt_date"] = pd.to_datetime(


In [39]:
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

In [40]:
df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

In [41]:
print("Mean Age:", np.mean(df["age"]))
print("Mean Distance:", np.mean(df["distance_km"]))
print("Mean Fees:", np.mean(df["fee_inr"]))
print("Mean Lead Days:", np.mean(df["lead_days"]))

Mean Age: 48.173737373737374
Mean Distance: 9.50090909090909
Mean Fees: 635.7141414141414
Mean Lead Days: 11.819191919191919


In [42]:
booked_more_than_14 = np.sum(
    df["lead_days"] > 14
)

print(
    "Appointments booked more than 14 days in advance:",
    booked_more_than_14
)

Appointments booked more than 14 days in advance: 296


In [43]:
mean_distance = np.mean(df["distance_km"])

std_distance = np.std(df["distance_km"])

df["distance_zscore"] = (
    df["distance_km"] - mean_distance
) / std_distance

print(df[["distance_km", "distance_zscore"]].head())

   distance_km  distance_zscore
0         18.0         1.380349
1         12.5         0.487087
2          4.1        -0.877169
3          7.5        -0.324970
4         19.0         1.542761


In [44]:
total_appointments = len(df)

missed = np.sum(df["no_show"])

attended = total_appointments - missed

no_show_rate = (
    missed / total_appointments
) * 100

print("Total Appointments:", total_appointments)
print("Missed Appointments:", missed)
print("Attended Appointments:", attended)
print("No-Show Rate:", round(no_show_rate, 1), "%")

Total Appointments: 990
Missed Appointments: 218
Attended Appointments: 772
No-Show Rate: 22.0 %


In [45]:
df.to_csv("cleaned_appointments.csv", index=False)

In [46]:
import pandas as pd
import numpy as np
import requests
import time

In [47]:
clinics = pd.read_csv("/content/clinics.csv")

print(clinics.head())
print(clinics.columns.tolist())

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


In [48]:
clinics["temp_c"] = np.nan

url = "https://api.open-meteo.com/v1/forecast"

for index, row in clinics.iterrows():

    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
        "timezone": "auto"
    }

    for attempt in range(2):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=20
            )

            response.raise_for_status()

            data = response.json()

            temperature = data["hourly"]["temperature_2m"][0]

            clinics.loc[index, "temp_c"] = temperature

            break

        except (requests.RequestException, KeyError,
                IndexError, TypeError, ValueError) as e:

            print(f"Clinic {index}: Attempt {attempt + 1} failed: {e}")

            if attempt == 0:
                time.sleep(2)

    time.sleep(0.2)

print(clinics[["temp_c"]])

   temp_c
0    21.6
1    21.8
2    21.6
3    16.9
4    16.9
5    29.1
6    19.0
7    16.4


In [49]:
clinics.to_csv("clinics_enriched.csv", index=False)

In [50]:
df = df.sort_values("appointment_id")

df = df.reset_index(drop=True)

In [51]:
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

In [52]:
print(df.isnull().sum())

appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
distance_zscore    0
dtype: int64


In [53]:
print("Duplicate appointment IDs:",
      df["appointment_id"].duplicated().sum())

Duplicate appointment IDs: 0


In [54]:
print("Final shape:", df.shape)

print(df.head())

Final shape: (990, 17)
  appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1          A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2          A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3          A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   
4          A0005       P251       C01  2024-11-23  2025-01-02  64.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  distance_zscore  
0             

In [55]:
if "distance_zscore" in df.columns:
    df = df.drop(columns=["distance_zscore"])

In [56]:
print("Final shape:", df.shape)

print(df.head())

Final shape: (990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1          A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2          A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3          A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   
4          A0005       P251       C01  2024-11-23  2025-01-02  64.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show    

In [57]:
df.to_csv("appointments_clean.csv", index=False)

In [59]:
import pandas as pd
import numpy as np

def sql_value(value):

    if pd.isna(value):
        return "NULL"

    if isinstance(value, (pd.Timestamp,)):
        return "'" + value.strftime("%Y-%m-%d") + "'"

    if isinstance(value, (np.integer, int)):
        return str(int(value))

    if isinstance(value, (np.floating, float)):
        return str(float(value))

    if isinstance(value, (bool, np.bool_)):
        return str(int(value))

    return "'" + str(value).replace("'", "''") + "'"


with open("load_data.sql", "w") as file:

    # Insert clinics first
    for _, row in clinics.iterrows():

        columns = ", ".join(clinics.columns)

        values = ", ".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO clinics ({columns}) VALUES ({values});\n"
        )

    # Insert appointments next
    for _, row in df.iterrows():

        columns = ", ".join(df.columns)

        values = ", ".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO appointments ({columns}) VALUES ({values});\n"
        )

In [60]:
with open("load_data.sql", "r") as file:
    sql_lines = file.readlines()

print("Total SQL INSERT statements:", len(sql_lines))

Total SQL INSERT statements: 998


In [61]:
print(clinics.columns.tolist())

['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']


In [64]:
import pandas as pd

appointments = pd.read_csv("appointments_clean.csv")

print("Total appointments:", len(appointments))
print("Total columns:", len(appointments.columns))
print("Unique IDs:", appointments["appointment_id"].nunique())
print("Missing values:", appointments.isna().sum().sum())

with open("load_data.sql", "r") as f:
    lines = f.readlines()

print("SQL INSERT statements:", len(lines))

Total appointments: 990
Total columns: 16
Unique IDs: 990
Missing values: 0
SQL INSERT statements: 998


In [65]:
from google.colab import files

files.download("load_data.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
from google.colab import files

files.download("load_data.sql")
files.download("appointments_clean.csv")
files.download("clinics_enriched.csv")
files.download("rejected_rows.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [67]:
import pandas as pd

appointments = pd.read_csv("appointments_clean.csv")
clinics = pd.read_csv("clinics_enriched.csv")

merged = appointments.merge(clinics, on="clinic_id")

result = merged.groupby(
    ["clinic_name", "city"]
).agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
).reset_index()

result["no_show_pct"] = (
    100 * result["no_shows"] / result["appointments"]
).round(1)

display(result.sort_values("no_show_pct", ascending=False))

,clinic_name,city,appointments,no_shows,no_show_pct
1,Gomti Nagar Care Point,Lucknow,90,30,33.3
2,Hadapsar Care Centre,Pune,143,39,27.3
4,Malviya Nagar Clinic,Jaipur,142,38,26.8
3,Kothrud Family Clinic,Pune,160,36,22.5
7,Vijay Nagar Clinic,Indore,101,21,20.8
0,Edappally Medical Centre,Kochi,142,23,16.2
5,Sitabuldi Health Point,Nagpur,82,13,15.9
6,Vaishali Nagar Clinic,Jaipur,130,18,13.8
